# Spam III: Term frequency, TF-IDF and vocabulary selection

Raw counts have two problems:

* **Frequent words dominate.** `you`, `to`, `the` occur everywhere and carry little information about the class.
* **Long messages dominate.** A message that repeats a word 10 times is not 10 times more spammy.

We fix both with **TF-IDF** (term frequency $\times$ inverse document frequency), implement it from scratch, check it against
scikit-learn, visualise it, and use it inside Naive Bayes.

In [ ]:
import numpy as np
import scipy.sparse as sp
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_selection import chi2
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from wordcloud import WordCloud

import spamlib as sl

plt.rcParams["figure.dpi"] = 100

texts, y = sl.load()
X_txt_train, X_txt_test, y_train, y_test = sl.split(texts, y)
vocab = sl.build_vocab(X_txt_train)
index = {w: i for i, w in enumerate(vocab)}
C_train = sl.count_matrix(X_txt_train, index)  # raw counts
C_test = sl.count_matrix(X_txt_test, index)
print(C_train.shape)

## 1. Term frequency: three ways to count

For a term $t$ in document $d$ with raw count $c_{t,d}$:

| variant | formula | effect |
|:--|:--|:--|
| raw | $c_{t,d}$ | long documents dominate |
| relative | $c_{t,d}/\sum_{t'}c_{t',d}$ | removes the length effect |
| sublinear | $1+\log c_{t,d}$ (if $c>0$) | the 10th repetition matters less than the 2nd |

Logarithms again: *diminishing returns* is exactly what a log does.

In [ ]:
def tf_sublinear(C):
    C = C.tocsr().copy()
    C.data = 1.0 + np.log(C.data)
    return C


def tf_relative(C):
    totals = np.asarray(C.sum(axis=1)).ravel()
    totals[totals == 0] = 1.0
    return sp.diags(1.0 / totals) @ C


for c in (1, 2, 5, 10):
    print(f"count {c:2d}: sublinear tf = {1 + np.log(c):.2f}")

## 2. Inverse document frequency

A word that occurs in *almost every* message tells us nothing; a word that occurs in a *few* messages is a strong signal.
With $N$ messages and $\text{df}(t)$ messages containing $t$:

$$
\text{idf}(t)=\ln\frac{1+N}{1+\text{df}(t)}+1
$$

(the "smooth" variant used by scikit-learn: the `+1` inside avoids a division by zero, the final `+1` keeps words that occur
everywhere from being erased completely). The logarithm makes the weight grow *slowly* with rarity.

In [ ]:
N = C_train.shape[0]
df = np.asarray((C_train > 0).sum(axis=0)).ravel()
idf = np.log((1 + N) / (1 + df)) + 1

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(df, idf, s=6, alpha=0.5)
ax.set(xscale="log", xlabel="document frequency df(t)", ylabel="idf(t)")
plt.tight_layout()
plt.show()

order = np.argsort(idf)
print("lowest idf  (most common):", [vocab[i] for i in order[:10]])
print("highest idf (rarest)     :", [vocab[i] for i in order[-5:]])

## 3. TF-IDF matrix from scratch

$$
\text{tfidf}(t,d)=\text{tf}(t,d)\cdot\text{idf}(t),\qquad
\mathbf{x}_d\leftarrow\frac{\mathbf{x}_d}{\lVert\mathbf{x}_d\rVert_2}
$$

The final **L2 normalisation** puts every message on the unit sphere, so length no longer matters.

In [ ]:
def tfidf(C, idf, sublinear=True):
    T = tf_sublinear(C) if sublinear else C.copy()
    T = T @ sp.diags(idf)
    norms = np.sqrt(np.asarray(T.multiply(T).sum(axis=1)).ravel())
    norms[norms == 0] = 1.0
    return sp.diags(1.0 / norms) @ T


T_train = tfidf(C_train, idf)
T_test = tfidf(C_test, idf)

Check against the reference implementation (same vocabulary, same variant):

In [ ]:
ref = TfidfTransformer(sublinear_tf=True).fit(C_train)
diff = abs(ref.transform(C_test) - T_test).max()
print("max |ours - sklearn| =", diff)

## 4. Look at the data: word clouds

Top row: the **class mean** of the TF-IDF vectors. Function words (`i`, `to`, `you`, `the`) still win, because they occur in
almost every message of the class: IDF shrinks them, it does not remove them. Bottom row: the *difference* between the class
mean and the mean of the other class. Now the words that **discriminate** stand out, which is what a classifier (and an
attacker) cares about.

In [ ]:
mean_spam = np.asarray(T_train[y_train == 1].mean(axis=0)).ravel()
mean_ham = np.asarray(T_train[y_train == 0].mean(axis=0)).ravel()

panels = [
    ("ham: class mean", mean_ham),
    ("spam: class mean", mean_spam),
    ("ham: mean - other class", np.clip(mean_ham - mean_spam, 0, None)),
    ("spam: mean - other class", np.clip(mean_spam - mean_ham, 0, None)),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (title, weights) in zip(axes.ravel(), panels):
    freqs = {w: float(v) for w, v in zip(vocab, weights) if v > 0}
    wc = WordCloud(background_color="white", width=600, height=350, max_words=80, random_state=42)
    ax.imshow(wc.generate_from_frequencies(freqs), interpolation="bilinear")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Naive Bayes on TF-IDF

Multinomial NB expects counts, but it only needs *non-negative feature weights*: TF-IDF acts as **fractional counts**
(Rennie et al., 2003). *Complement NB* estimates each class from the *other* class's data and is more robust with
skewed classes.

In [ ]:
results = {}
for name, (A, B) in {"counts": (C_train, C_test), "tf-idf": (T_train, T_test)}.items():
    for cls in (MultinomialNB, ComplementNB):
        m = cls(alpha=0.1).fit(A, y_train)
        score = (
            m.predict_log_proba(B)[:, 1] - m.predict_log_proba(B)[:, 0]
            if cls is MultinomialNB
            else m.predict_proba(B)[:, 1]
        )
        results[f"{cls.__name__} on {name}"] = sl.scores(y_test, m.predict(B), score)
sl.show(results)

## 6. Vocabulary selection

Do we need 3,000 words? Rank the words with a $\chi^2$ test between *word occurs* and *class* and keep only the best $k$.
The remaining features are the **most informative**, and they are also the ones an attacker has to manipulate.

In [ ]:
chi, _ = chi2(T_train, y_train)
chi = np.nan_to_num(chi)
rank = np.argsort(-chi)

ks = [10, 25, 50, 100, 250, 500, 1000, len(vocab)]
f1s = []
for k in ks:
    cols = rank[:k]
    m = MultinomialNB(alpha=0.1).fit(T_train[:, cols], y_train)
    f1s.append(sl.scores(y_test, m.predict(T_test[:, cols]))["f1"])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ks, f1s, "o-")
ax.set(xscale="log", xlabel="number of selected words k", ylabel="F1 (spam)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("top-15 words by chi2:", [vocab[i] for i in rank[:15]])

## Exercises

1. Use `sublinear=False` and compare. Is the difference significant? (Hint: repeat with 5 different seeds in `sl.split`.)
2. Add **bigrams** (`"call now"`, `"free entry"`) to the vocabulary. What happens to the vocabulary size, the density and the score?
3. Repeat the selection experiment with `min_df=1`. Which words appear in the top-15? Why are they suspicious?
4. **Attacker.** Rare words have a high idf. Explain why appending *rare* words to a spam message can *reduce* the weight of the
   words that reveal it, once the vector is L2-normalised.